# WignerFlow: resumable 20 ns stability experiment (A100)
Select an **A100 GPU** runtime, edit parameters, then Run all. This is an experiment, not a claim of 20 ns stability. The observed argon baseline crashes around 865 × 50 fs ≈ 43 ps. All durable artifacts live on Drive. Completed stages skip; interrupted data are regenerated, training resumes from ckpt.pt, and rollout resumes from its last committed chunk. Training does not restore RNG/DataLoader state; rollout saves Python/NumPy/CPU/CUDA RNG and float64 position history.

Use a new RUN_NAME for changed inputs/configuration. Setup must run on every new Colab runtime. Drive persistence is subject to Drive/FUSE durability; retain backups. A damaged HDF5 file fails loudly rather than silently starting over. Long transport validation is bounded by max_lag but still expensive. Single-start results need independent starts/seeds before acceptance.


In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, subprocess, json, time, shutil, random, math, uuid

drive.mount('/content/drive')
SYSTEM = 'argon'  # argon | nacl_proof | li_pff | na_pff
DATA_SOURCE = Path('/content/drive/MyDrive/WignerFlow_inputs/li_pff')
TARGET_NS = 20
STRIDE = 10  # integrator steps, NOT fs; must be divisible by input save_every
CHUNK_NS = 0.25
RUN_NAME = SYSTEM + '_20ns_stable_v1'
# External inputs: list independent trajectories explicitly (at least five).
SOURCE_FILES = []  # e.g. ['run0.xtc', ..., 'run4.xtc']; relative to DATA_SOURCE
TOPOLOGY = 'system.pdb'
SYSTEM_XML = 'system.xml'  # exact OpenMM force field/system for same-system benchmark
IMPORT_FORMAT = 'mdtraj'  # mdtraj | ase | generic
INPUT_DT_FS = 1.0
INPUT_SAVE_EVERY = 10
INPUT_UNIT = 'nm'
VELOCITY_UNIT = 'nm/ps'
VELOCITY_FILES = []  # one actual velocity NPY/NPZ/HDF5 per source for DCD/XTC
KEEP_DRUDE = False  # importer --keep-drude; positive masses required downstream
CHARGES = None  # CLI supports element -> charge only; insufficient for molecular partial charges
TEMPERATURE_K = 95.0 if SYSTEM == 'argon' else (1200.0 if SYSTEM == 'nacl_proof' else 300.0)
MIN_DISTANCE_A = 0.5  # customize for chemistry/Drude; not a bonded-topology check
SAMPLING_STEPS = 10
assert SYSTEM in {'argon', 'nacl_proof', 'li_pff', 'na_pff'}
assert TARGET_NS > 0 and STRIDE > 0 and CHUNK_NS > 0
RUN = Path('/content/drive/MyDrive/WignerFlow_runs') / RUN_NAME
RUN.mkdir(parents=True, exist_ok=True)
REPO = Path('/content/WignerFlow')
BRANCH = 'feat/20ns-stability'
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['MKL_NUM_THREADS'] = '2'
os.environ['OPENMM_CPU_THREADS'] = '2'
os.environ.pop('PYTHONPATH', None)

def atomic_json(path, obj):
    tmp = Path(str(path) + '.tmp')
    tmp.write_text(json.dumps(obj, indent=2, allow_nan=False))
    os.replace(tmp, path)

def run(command, log_name='commands.log'):
    command = list(map(str, command))
    print(' '.join(command), flush=True)
    start = time.perf_counter()
    log_path = RUN / log_name
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open('a') as log:
        with subprocess.Popen(command, cwd=REPO if REPO.exists() else '/content',
                              stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                print(line, end='')
                log.write(line)
                log.flush()
            status = proc.wait()
    with (RUN / 'commands.jsonl').open('a') as f:
        f.write(json.dumps(dict(command=command, wall_s=time.perf_counter()-start, exit=status))+'\n')
    if status:
        raise RuntimeError(f'Command failed with exit {status}; see {log_path}')


In [ ]:
# Runtime setup is deliberately repeated after disconnects; persistent stages below skip.
if not REPO.exists():
    run(['git', 'clone', '--branch', BRANCH, '--single-branch',
         'https://github.com/yaniguan/WignerFlow.git', REPO], 'setup.log')
run(['git', 'checkout', BRANCH], 'setup.log')
os.chdir(REPO)
run([sys.executable, '-m', 'pip', 'install', '-e', '.[simulation,train]', 'h5py', 'ase', 'mdtraj'], 'setup.log')
run([sys.executable, '-m', 'pip', 'install', 'openmm[cuda12]>=8.1'], 'setup.log')
import torch, numpy as np, h5py, yaml, openmm
assert torch.cuda.is_available(), 'Select an A100 GPU runtime'
assert 'A100' in torch.cuda.get_device_name(0), torch.cuda.get_device_name(0)
assert 'CUDA' in [openmm.Platform.getPlatform(i).getName() for i in range(openmm.Platform.getNumPlatforms())]
torch.set_num_threads(2)
if not (RUN / 'environment.txt').exists():
    run([sys.executable, '-m', 'pip', 'freeze'], 'environment.txt')
    run(['git', 'rev-parse', 'HEAD'], 'source_commit.txt')
from equitraj.profiling import GpuSampler, write_summary
PROF = RUN / 'profiling'
PROF.mkdir(exist_ok=True)
# Keep per-session utilization CSVs: GpuSampler opens files in write mode.
def sampler(stage):
    return GpuSampler(str(PROF / f'{stage}_{uuid.uuid4().hex}.csv'), interval_s=1.0)


## Data and stability recipe
Argon uses the full-data generator (ten 0.5 ns trajectories). NaCl uses illustrative LJ+Coulomb PME on CUDA; it is not a validated molten-salt force field. PFF imports user trajectories split by trajectory. DCD/XTC lacks velocities: supply separate velocity arrays. The importer can estimate missing velocities, but this notebook rejects estimated velocities for production training. Dropping Drudes does not redistribute their masses; retaining Drudes does not add polarization physics to the model. Charges/polarizabilities/topology metadata are not model features on this branch. A molecular PFF result remains exploratory.


In [ ]:
DATA_DRIVE = RUN / 'data'
DATA_LOCAL = REPO / 'data' / RUN_NAME
DATA_DRIVE.mkdir(exist_ok=True)
DATA_LOCAL.mkdir(parents=True, exist_ok=True)
def complete(path):
    try:
        with h5py.File(path, 'r') as h:
            return bool(h.attrs.get('complete', False))
    except (OSError, FileNotFoundError):
        return False

if not (DATA_DRIVE / 'done.json').exists():
    if SYSTEM == 'argon':
        run([sys.executable, 'scripts/gen_data.py', '--config', 'configs/data/argon_full.yaml',
             '--platform', 'CUDA', '--out_dir', DATA_DRIVE], 'data.log')
        count = 10
    elif SYSTEM == 'nacl_proof':
        # Generator has no per-trajectory resume flag. Generate each missing trajectory
        # separately with a distinct seed, then promote it to the canonical filename.
        count = 5
        for i in range(count):
            dest = DATA_DRIVE / f'traj_{i:03d}.h5'
            if complete(dest):
                continue
            scratch = DATA_DRIVE / f'partial_{i:03d}'
            run([sys.executable, 'scripts/make_nacl_proof.py', '--out-dir', scratch,
                 '--platform', 'CUDA', '--n-traj', '1', '--seed', str(2026+i),
                 '--frames', '100001', '--save-every', '5', '--dt-fs', '1', '--overwrite'], 'data.log')
            os.replace(scratch / 'traj_000.h5', dest)
            for name in ['system.xml', 'system.pdb']:
                shutil.copy2(scratch / name, DATA_DRIVE / name)
        from equitraj.adapters import write_data_config
        write_data_config(DATA_DRIVE, count)
    else:
        assert len(SOURCE_FILES) >= 5, 'Provide at least five independent source trajectories'
        assert STRIDE % INPUT_SAVE_EVERY == 0
        count = len(SOURCE_FILES)
        from equitraj.adapters import write_data_config
        for i, name in enumerate(SOURCE_FILES):
            dest = DATA_DRIVE / f'traj_{i:03d}.h5'
            if complete(dest):
                continue
            scratch = DATA_DRIVE / f'partial_{i:03d}'
            cmd = [sys.executable, 'scripts/import_md.py', DATA_SOURCE / name,
                   '--out-dir', scratch, '--format', IMPORT_FORMAT, '--unit', INPUT_UNIT,
                   '--velocity-unit', VELOCITY_UNIT, '--dt-fs', str(INPUT_DT_FS),
                   '--save-every', str(INPUT_SAVE_EVERY), '--system', SYSTEM, '--overwrite']
            if IMPORT_FORMAT == 'mdtraj':
                cmd += ['--topology', DATA_SOURCE / TOPOLOGY]
                assert len(VELOCITY_FILES) == count, 'DCD/XTC needs actual separate velocities'
            if VELOCITY_FILES:
                assert len(VELOCITY_FILES) == count
                cmd += ['--velocity-source', DATA_SOURCE / VELOCITY_FILES[i]]
            if KEEP_DRUDE:
                cmd += ['--keep-drude']
            run(cmd, 'data.log')
            os.replace(scratch / 'traj_000.h5', dest)
        write_data_config(DATA_DRIVE, count, {'train': list(range(count-2)), 'val': [count-2], 'test': [count-1]})
    for i in range(count):
        assert complete(DATA_DRIVE / f'traj_{i:03d}.h5')
    atomic_json(DATA_DRIVE / 'done.json', {'count': count})

for src in DATA_DRIVE.glob('traj_*.h5'):
    with h5py.File(src, 'r') as h:
        assert h.attrs['complete'] and not h.attrs.get('velocities_estimated', False)
        assert np.all(h['masses'][:] > 0), 'Massless particles are unsupported; preprocess explicitly'
        assert STRIDE % int(h.attrs['save_every']) == 0
    dst = DATA_LOCAL / src.name
    if not dst.exists() or dst.stat().st_size != src.stat().st_size:
        shutil.copy2(src, dst)
shutil.copy2(DATA_DRIVE / 'config.yaml', DATA_LOCAL / 'config.yaml')
CONFIG = RUN / 'stability_runtime.yaml'
if not CONFIG.exists():
    cfg = yaml.safe_load((REPO / 'configs/argon/B_S3a_stable.yaml').read_text())
    # Use the A100 stability recipe for all systems, rather than the 600-step CPU smoke recipe.
    cfg.update(data_dir=str(DATA_LOCAL), out_dir=str(RUN / 'train'), stride_steps=STRIDE)
    cfg['train']['ckpt_minutes'] = 10
    CONFIG.write_text(yaml.safe_dump(cfg))
cfg = yaml.safe_load(CONFIG.read_text())
assert cfg['stride_steps'] == STRIDE and cfg['data_dir'] == str(DATA_LOCAL)
# Check local interaction range against reciprocal-plane box heights (also triclinic).
for src in DATA_LOCAL.glob('traj_*.h5'):
    with h5py.File(src, 'r') as h:
        for cell in h['box'][::max(1, len(h['box'])//100)]:
            heights_A = 10 / np.linalg.norm(np.linalg.inv(cell), axis=0)
            assert cfg['model']['cutoff'] < 0.5*heights_A.min(), 'Box too small for cutoff'
print(yaml.safe_dump(cfg))


## Train (resumes ckpt.pt)
The saved config enables unroll 1→2→4, annealed noise, repulsion prior, overlap penalty, and longer stability-first checkpoint selection. best_stable.pt is chosen on validation only. Keep the test reference out of selection. A completed done.json skips training.


In [ ]:
if not (RUN / 'train/done.json').exists():
    with sampler('training') as gpu:
        run([sys.executable, '-m', 'equitraj.train', '--config', CONFIG, '--device', 'cuda'], 'training.log')
    write_summary(PROF, {'full_training_gpu': gpu.summary()})
CKPT = RUN / 'train/best_stable.pt'
assert CKPT.exists()


## Required profiling (same physical system)
The benchmark measures synchronized, batch-one autoregressive throughput and OpenMM integration throughput at their respective physical step sizes. It excludes setup/I/O and does not establish accuracy. The full rollout below records wall time including monitoring and Drive writes separately. For PFF supply the exact system XML and PDB in matching atom order; this stage stops if the benchmark cannot represent the training particles. profile_run saves exclusive data/forward/backward/optimizer/neighbor-list timings, peak CUDA memory, utilization samples, a short Chrome trace, operator table, and summary.json. A completed profiling/done.json skips profiling; partial output alone is not completion.


In [ ]:
if not (PROF / 'done.json').exists():
    bench_path = PROF / 'openmm.json'
    if not bench_path.exists():
        system_flags = ['--system', 'argon'] if SYSTEM == 'argon' else [
            '--xml', DATA_DRIVE / 'system.xml' if SYSTEM == 'nacl_proof' else DATA_SOURCE / SYSTEM_XML,
            '--pdb', DATA_DRIVE / 'system.pdb' if SYSTEM == 'nacl_proof' else DATA_SOURCE / TOPOLOGY]
        # run() preserves exact stdout as an append-only log. Parse only this fresh log.
        bench_log = 'profiling/bench_' + uuid.uuid4().hex + '.log'
        run([sys.executable, 'scripts/bench_openmm.py', *system_flags,
             '--checkpoint', CKPT, '--platform', 'CUDA', '--device', 'cuda',
             '--dt-fs', str(5 if SYSTEM == 'argon' else (1 if SYSTEM == 'nacl_proof' else INPUT_DT_FS)),
             '--temperature', str(TEMPERATURE_K), '--steps', '10000', '--model-steps', '100',
             '--sampling-steps', str(SAMPLING_STEPS)], bench_log)
        atomic_json(bench_path, json.loads((RUN / bench_log).read_text()))
    run([sys.executable, 'scripts/profile_run.py', '--config', CONFIG, '--ckpt', CKPT,
         '--out', PROF, '--device', 'cuda', '--train-steps', '20', '--train-warmup', '3',
         '--rollout-steps', '100', '--rollout-batch', '1', '--sampling-steps', str(SAMPLING_STEPS),
         '--profile', 'train', '--profile-steps', '3', '--gpu-interval', '1',
         '--openmm-json', bench_path], 'profiling/profile.log')
    for name in ['summary.json', 'trace.json', 'key_averages.txt', 'gpu_samples.csv']:
        assert (PROF / name).exists(), name
    atomic_json(PROF / 'done.json', {'complete': True})
print((PROF / 'summary.json').read_text())


## GPU speed benchmark (A100)
Synthetic FCC systems, random weights, 128–16384 atoms. This optional sweep records failures and precision/compile fallbacks, compares 1 vs 10 flow evaluations with OpenMM CUDA, and saves one fast-step profiler trace. It does not establish model accuracy. A100 has no native float8 Tensor Cores.


In [ ]:
# This synthetic random-init sweep measures compute only; A100 fp8 may fall back.
subprocess.run([sys.executable, '-m', 'pip', 'install', 'torchao'], check=True)
SPEED = RUN / 'profiling' / 'speed'
SPEED.mkdir(parents=True, exist_ok=True)
assert torch.cuda.is_available()
print('GPU:', torch.cuda.get_device_name())
run([sys.executable, 'scripts/bench_speed.py', '--device', 'cuda', '--platform', 'CUDA',
     '--sizes', '128', '512', '2048', '8192', '16384',
     '--precisions', 'fp32', 'bf16', 'fp8', '--sampling-steps', '1', '10',
     '--replicas', '1', '4', '--compile', '0', '1', '--neighbors', 'verlet',
     '--steps', '20', '--warmup', '3', '--stride-fs', '50', '--dt-fs', '2',
     '--out', SPEED / 'results.jsonl', '--trace', SPEED / 'trace'],
    'profiling/speed/sweep.log')
rows = [json.loads(line) for line in (SPEED / 'results.jsonl').read_text().splitlines()]
write_summary(PROF, {'speed': {'results': str(SPEED / 'results.jsonl'),
                             'trace': str(SPEED / 'trace' / 'trace.json'),
                             'gpu': torch.cuda.get_device_name(),
                             'failures': [r for r in rows if 'error' in r],
                             'rows': rows}})
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 5))
for precision in ['fp32', 'bf16', 'fp8']:
    for sampling in [1, 10]:
        selected = sorted([r for r in rows if r.get('engine') == 'wignerflow'
                           and r.get('precision') == precision and r.get('sampling_steps') == sampling
                           and r.get('replicas') == 1 and not r.get('compile') and 'error' not in r],
                          key=lambda r: r['atoms'])
        if selected:
            effective = ','.join(sorted(set(r['effective_precision'] for r in selected)))
            ax.plot([r['atoms'] for r in selected], [r['ns_per_day'] for r in selected],
                    'o-', label=f'{precision} (effective {effective}), {sampling} samples')
omm = sorted([r for r in rows if r['engine'] == 'openmm' and 'error' not in r], key=lambda r: r['atoms'])
if omm:
    ax.plot([r['atoms'] for r in omm], [r['ns_per_day'] for r in omm], 'k--', label='OpenMM CUDA, 2 fs')
ax.set(xscale='log', yscale='log', xlabel='Atoms', ylabel='ns/day per replica',
       title='Synthetic compute throughput; random model, 50 fs stride')
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(SPEED / 'ns_per_day.png', dpi=160)
plt.show()


## Chunked rollout and transactional resume
Each chunk advances at most 0.25 ns and checks every step. The monitor captures the true float64 accumulated position history because rollout() only returns float32 saved frames. The HDF5 contains an initial frame and every stable predicted frame in repo units (nm, nm/ps, ps); crash frames are excluded. At a crash, record its global step and the preceding stable length, then stop. Per-chunk diagnostics include temperature/KE drift and offending atoms. No Hamiltonian is available for total-energy drift.

Commit order: append + flush/close HDF5, then atomically replace state.pt. On restart truncate uncommitted HDF5 tails to state.frames; replay the interrupted chunk from saved RNG. Monitor history is chunk-local and archived with global step numbers, while validation fits drift across the full stable trajectory. Atomic replacement is logical recovery, not a guarantee against remote storage corruption. Fixed box, model only, no thermostat/barostat. Changing checkpoint/parameters on an existing run is rejected.


In [ ]:
from equitraj.evaluation import load_model
from equitraj.rollout import rollout
from equitraj.monitor import StabilityMonitor, kinetic_energy
import hashlib
model, _, ds, train_step = load_model(str(RUN / 'train'), 'best_stable.pt', torch.device('cuda'),
                                      'test', str(DATA_LOCAL))
DT = ds.stride_fs
TOTAL = math.ceil(TARGET_NS * 1e6 / DT)
CHUNK = max(1, round(CHUNK_NS * 1e6 / DT))
ROLL = RUN / 'rollout'
ROLL.mkdir(exist_ok=True)
STATE, H5 = ROLL / 'state.pt', ROLL / 'trajectory.h5'
fingerprint = hashlib.sha256(CKPT.read_bytes()).hexdigest()
identity = dict(checkpoint_sha256=fingerprint, stride_fs=DT, total=TOTAL,
                sampling_steps=SAMPLING_STEPS, min_distance_A=MIN_DISTANCE_A,
                temperature_K=TEMPERATURE_K)

def save_state(s):
    torch.save(s, str(STATE)+'.tmp')
    os.replace(str(STATE)+'.tmp', STATE)

def rng_state():
    return dict(python=random.getstate(), numpy=np.random.get_state(), cpu=torch.get_rng_state(),
                cuda=torch.cuda.get_rng_state_all())

def restore_rng(r):
    random.setstate(r['python'])
    np.random.set_state(r['numpy'])
    torch.set_rng_state(r['cpu'])
    torch.cuda.set_rng_state_all(r['cuda'])

if STATE.exists():
    s = torch.load(STATE, map_location='cpu', weights_only=False)
    assert s['identity'] == identity, 'Changed run identity: use a new RUN_NAME'
else:
    assert not H5.exists(), 'Orphan HDF5: inspect before recovery'
    pos, vel, bx = ds[0]
    random.seed(0); np.random.seed(0); torch.manual_seed(0)
    s = dict(identity=identity, x=pos[:model.k][None].double(), v=vel[:model.k][None],
             box=bx[None] if ds.periodic else None, step=0, frames=1, stable_steps=0,
             crashed=False, rng=rng_state(), wall_s=0.0)
    save_state(s)

names = ['positions', 'velocities', 'box', 'time', 'kinetic_energy', 'potential_energy', 'forces']
if not H5.exists():
    assert s['step'] == 0
    from equitraj.trajfile import TrajWriter
    w = TrajWriter(H5, len(ds.masses), ds.masses.numpy(), ds.meta['elements'],
                   dict(system=SYSTEM, periodic=ds.periodic, complete=False, dt_fs=DT,
                        save_every=1, positions_unwrapped=True, ensemble='model_only'))
    box_nm = s['box'][0].numpy()/10 if s['box'] is not None else np.zeros((3,3))
    w.add(s['x'][0,-1].numpy()/10, s['v'][0,-1].numpy()*100,
          np.full((len(ds.masses),3), np.nan), np.nan,
          kinetic_energy(s['v'][:,-1], ds.masses).item(), box_nm, 0.0)
    w.close()
with h5py.File(H5, 'r+') as h:
    for name in names:
        assert len(h[name]) >= s['frames'], f'Missing committed frames in {name}'
        h[name].resize(s['frames'], axis=0)
    if not (ROLL / 'done.json').exists():
        h.attrs['complete'] = False
    h.flush()

class CaptureMonitor(StabilityMonitor):
    def __init__(self, state):
        super().__init__(ds.types, ds.masses, box=state['box'], min_dist=MIN_DISTANCE_A,
                         t_max=3*TEMPERATURE_K, stride_fs=DT)
        self.offset = state['step']
        self.xctx, self.vctx = state['x'].clone(), state['v'].clone()
    def check(self, step, x, v):
        # Called at every step; preserve float64 context instead of reconstructing from float32 outputs.
        self.xctx = torch.cat([self.xctx[:,1:], x.detach().cpu()[:,None]], dim=1)
        self.vctx = torch.cat([self.vctx[:,1:], v.detach().cpu()[:,None]], dim=1)
        return super().check(self.offset + step, x, v)

def clean(obj):
    if isinstance(obj, dict): return {k: clean(v) for k,v in obj.items()}
    if isinstance(obj, list): return [clean(v) for v in obj]
    if isinstance(obj, float) and not math.isfinite(obj): return None
    return obj

if not (ROLL / 'done.json').exists():
    restore_rng(s['rng'])
    with sampler('rollout') as gpu:
        while s['step'] < TOTAL and not s['crashed']:
            started = time.perf_counter()
            monitor = CaptureMonitor(s)
            out = rollout(model, s['x'], s['v'], ds.types, s['box'], ds.masses,
                          min(CHUNK, TOTAL-s['step']), sampling_steps=SAMPLING_STEPS,
                          save_every=1, check_every=1, monitor=monitor)
            n = out['steps_done']
            failed = int(out['crash_step'][0]) >= 0
            good = n-1 if failed else n
            x = out['positions'][:good,0].numpy()
            v = out['velocities'][:good,0].numpy()
            box_nm = s['box'][0].numpy()/10 if s['box'] is not None else np.zeros((3,3))
            ke = kinetic_energy(torch.from_numpy(v), ds.masses).numpy() if good else np.empty(0)
            data = dict(positions=x/10, velocities=v*100,
                        box=np.repeat(box_nm[None], good, axis=0),
                        time=(s['step']+np.arange(1,good+1))*DT/1000,
                        kinetic_energy=ke, potential_energy=np.full(good,np.nan),
                        forces=np.full(x.shape,np.nan))
            with h5py.File(H5, 'r+') as h:
                for name, values in data.items():
                    h[name].resize(s['frames']+good, axis=0)
                    h[name][s['frames']:] = values
                h.flush()
            end = s['step']+n
            atomic_json(ROLL / f'diagnostics_{end:09d}.json', clean(out['diagnostics']))
            s.update(x=monitor.xctx, v=monitor.vctx, step=end, frames=s['frames']+good,
                     stable_steps=end-1 if failed else end, crashed=failed,
                     rng=rng_state(), wall_s=s['wall_s']+time.perf_counter()-started)
            save_state(s)
            print(f'{end}/{TOTAL} steps; stable {s["stable_steps"]*DT/1e6:.6f} ns', flush=True)
    write_summary(PROF, {'full_rollout_gpu': gpu.summary()})
    with h5py.File(H5, 'r+') as h:
        h.attrs['complete'] = True  # complete artifact, even when target failed
        h.attrs['target_reached'] = not s['crashed'] and s['step'] >= TOTAL
        h.attrs['stable_ns'] = s['stable_steps']*DT/1e6
    atomic_json(ROLL / 'done.json', dict(step=s['step'], stable_ns=s['stable_steps']*DT/1e6,
                crashed=s['crashed'], target_ns=TARGET_NS, wall_s=s['wall_s'],
                steps_per_s=s['step']/s['wall_s'], ns_per_day=s['step']*DT/1e6/s['wall_s']*86400))
print((ROLL / 'done.json').read_text())


## Held-out validation
validate_md reports pair RDFs, per-species MSD/diffusion, VACF, temperature drift, and stable duration. Conductivity is computed for NaCl using element charges. For PFF the CLI's element-charge mapping cannot represent atom-specific molecular charges: leave conductivity unavailable unless that mapping is physically valid; extend the validator with aligned per-atom charges before claiming electrolyte conductivity. Bound lag cost and choose a physically diffusive fit window before using transport estimates. A crash at the first step leaves no dynamical segment; record that explicitly.


In [ ]:
VALID = RUN / 'validation'
VALID.mkdir(exist_ok=True)
if not (VALID / 'done.json').exists():
    if s['stable_steps'] < 2:
        atomic_json(VALID / 'metrics.json', {'status': 'insufficient stable frames', 'stable_ns': s['stable_steps']*DT/1e6})
    else:
        # Subsample held-out reference to exactly the model cadence, in bounded blocks.
        REF = VALID / 'reference.h5'
        if not REF.exists():
            src = DATA_DRIVE / f'traj_{ds.traj_ids[0]:03d}.h5'
            tmp = Path(str(REF)+'.tmp')
            with h5py.File(src, 'r') as a, h5py.File(tmp, 'w') as b:
                for key, value in a.attrs.items(): b.attrs[key] = value
                b.attrs.update(dt_fs=DT, save_every=1, force_every=1, complete=True)
                for name in ['elements', 'masses']: b.create_dataset(name, data=a[name][:])
                indices = np.arange(0, len(a['positions']), ds.f)
                for name in names:
                    shape = (len(indices),)+a[name].shape[1:]
                    d = b.create_dataset(name, shape=shape, dtype=a[name].dtype)
                    for j in range(0,len(indices),1000):
                        ids = indices[j:j+1000]
                        d[j:j+len(ids)] = np.nan if name == 'forces' else a[name][ids]
            os.replace(tmp, REF)
        charges = {'Na': 1, 'Cl': -1} if SYSTEM == 'nacl_proof' else CHARGES
        cmd = [sys.executable, 'scripts/validate_md.py', H5, REF,
               '--max-lag', '200', '--n-bins', '100', '--temperature-K', str(TEMPERATURE_K),
               '--dof', str(3*len(ds.masses)-3), '--min-distance', str(MIN_DISTANCE_A),
               '--max-temperature', str(3*TEMPERATURE_K), '--json-out', VALID / 'metrics.json',
               '--markdown-out', VALID / 'metrics.md']
        if charges is not None: cmd += ['--charges', json.dumps(charges)]
        run(cmd, 'validation.log')
    atomic_json(VALID / 'done.json', {'complete': True, 'conductivity_charge_model':
                 'element mapping only; atom-specific PFF charges require extension'})
if (VALID / 'metrics.md').exists():
    from IPython.display import display, Markdown
    display(Markdown((VALID / 'metrics.md').read_text()))


## Binding validation placeholder (protein–ligand systems)
Supply atom ordering, ligand/pocket/backbone selections, bonds/angles, common reference pose, and matched held-out trajectories. Call equitraj.binding.compare_binding (Å units) for pocket-aligned ligand RMSD, internal RMSD, contact persistence, COM displacement, geometric H-bond contacts, and bond/angle distributions. Supply a force-field energy callable for interaction_energy and use exponential_average/mean_difference only as free-energy proxies with explicit beta/units, overlap checks, and uncertainty. These are not calibrated binding free energies. No binding result is generated for the four bulk-system options above.

Optional independent multistart validation uses the real CLI: scripts/evaluate.py --run <train> --ckpt best_stable.pt --data_dir <data> --split test --steps 1000 --starts 3 --seeds 5 --sampling_steps 10 --device cuda --out <eval>. This CLI is not resumable and its argon thresholds do not establish PFF accuracy; the long trajectory above uses the chunked wrapper.


In [ ]:
# Final results table. All required profiling output remains under RUN/profiling/.
p = json.loads((PROF / 'summary.json').read_text())
b = json.loads((PROF / 'openmm.json').read_text())
r = json.loads((ROLL / 'done.json').read_text())
write_summary(PROF, {'long_rollout': r, 'validation': str(VALID / 'metrics.json')})
rows = [
    ('system', SYSTEM), ('checkpoint train step', train_step), ('stride fs', DT),
    ('target autoregressive steps', TOTAL), ('stable ns', r['stable_ns']), ('crashed', r['crashed']),
    ('rollout compute steps/s', p['rollout'].get('steps_per_s')),
    ('rollout compute ns/day', p['rollout'].get('ns_per_day')),
    ('long rollout incl. monitor/Drive steps/s', r['steps_per_s']),
    ('long rollout incl. monitor/Drive ns/day', r['ns_per_day']),
    ('OpenMM steps/s', b['openmm']['steps_per_s']), ('OpenMM ns/day', b['openmm']['ns_per_day']),
    ('validation report', str(VALID / 'metrics.json')), ('profiling summary', str(PROF / 'summary.json'))]
print('| Result | Value |\n|---|---|')
for label, value in rows: print(f'| {label} | {value} |')
print('Training phases / peak GPU memory:', json.dumps(p.get('train', {}), indent=2))
print('Artifacts:', RUN)
